# Generation of Topologically Associated Domains and their Boundaries

## Description

This notebook builds **TAD boundaries (TADBs)** and TADB-enhanced cis windows from tissue/brain TAD calls. It runs in two conceptual stages: (i) *manage TAD redundancy* by recursively merging overlapping specific TADs (those whose overlap exceeds a cutoff) into a smaller set of generalized TADs, and (ii) *generate TADB-enhanced cis windows and extended TADBs* by deriving boundaries from the generalized + specific TADs and joining them with gene coordinates to produce per-gene cis windows.

> **Synthetic-data note.** This minimal working example runs on a small synthetic set of chr22 TAD blocks (`protocol_example.brain_TADs.txt`) and a chr22 gene table (`protocol_example.gene_start_end.tsv`, derived from the protocol gene-model GTF). It is for demonstration only and does not reproduce a full genome-wide TADB build.

## Input

- `--tad-input`: tab-separated TAD coordinates with columns `chr`, `start`, `end` (no header). Example: `input/tadb/protocol_example.brain_TADs.txt`.
- `--gene-coords`: gene coordinate table with columns `index`, `#chr`, `start`, `end`, `gene_id`, `gene_name` (derived from the gene-model GTF). Example: `input/tadb/protocol_example.gene_start_end.tsv`.
- `--cwd`: output directory (default `output/tadb`).
- `--overlap-cutoff`: minimum percent overlap to merge two specific TADs into one generalized TAD (default `80`).

## Output

- `generalized_TAD.tsv`: redundancy-managed (merged) generalized TADs (`chr`, `start`, `end`).
- `generalized_TADB.tsv`: generalized TAD boundaries with a `TADB` index.
- `TADB_enhanced_cis.bed`: per-gene TADB-enhanced cis windows (`#chr`, `start`, `end`, `gene_id`).
- `extended_TADB.bed`: extended TAD boundaries (`#chr`, `start`, `end`, `index`).

## Steps

**Step 1.** Run the consolidated `default` workflow. It manages TAD redundancy (recursive merge of overlapping TADs), derives generalized TAD boundaries, and builds the TADB-enhanced cis windows and extended TADBs in a single pass.

**Timing:** ~varies on typical compute infrastructure.

In [ ]:
sos run pipeline/generalized_TADB.ipynb default \
    --tad-input input/tadb/protocol_example.brain_TADs.txt \
    --gene-coords input/tadb/protocol_example.gene_start_end.tsv \
    --cwd output/tadb

## Command interface

In [ ]:
sos run pipeline/generalized_TADB.ipynb -h

## Workflow implementation

The `[global]` section declares parameters (inputs, output directory, the merge `overlap_cutoff`, and resource options; the empty `container` and the `container=container` task option are kept for parity with the protocol). The single `[default]` section runs the whole pipeline in R: it defines the TAD overlap/merge helpers (`find_TAD_overlap`, `merge_TADs`, `recursive_merge`), produces the generalized TADs, derives left/right boundaries against the hard-coded hg38 chromosome lengths, builds the generalized TADBs, and joins gene coordinates to write the cis-window and extended-TADB BED files. (The original notebook also contains a Python `gtf_to_start_end_bed` helper that derives the gene table from the GTF; here that table is supplied as the `--gene-coords` input.)

In [ ]:
[global]
# Brain/tissue TAD coordinates: a tab-separated file with columns chr, start, end (no header)
parameter: tad_input = path
# Gene coordinates table derived from the gene-model GTF (columns: index, #chr, start, end, gene_id, gene_name)
parameter: gene_coords = path
# Output directory
parameter: cwd = path("output/tadb")
# Minimum percent overlap for two specific TADs to be merged into one generalized TAD
parameter: overlap_cutoff = 80
# Container image (kept empty; we no longer use containers)
parameter: container = ''
# Resource options (kept for parity with the protocol)
parameter: job_size = 1
parameter: walltime = "20h"
parameter: mem = "40G"
parameter: numThreads = 1

In [ ]:
[default]
input: tad = tad_input, gene = gene_coords
output: gentad = f'{cwd}/generalized_TAD.tsv',
        gentadb = f'{cwd}/generalized_TADB.tsv',
        cis = f'{cwd}/TADB_enhanced_cis.bed',
        ext = f'{cwd}/extended_TADB.bed'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f'{step_name}_{_output["cis"]:bn}'
R: container=container, expand = "${ }", stderr = f'{_output["cis"]:n}.stderr', stdout = f'{_output["cis"]:n}.stdout'
    library(tidyverse)
    
    find_TAD_overlap <- function(x, inputDF){
        rowChr <- x['chr']
        rowStart <- as.numeric(x['start'])
        rowEnd <- as.numeric(x['end'])
        rowTADIndex <- x['TAD_index']
        TADsubset <- inputDF %>% filter(chr == rowChr)
        TADsubset$start <- as.numeric(TADsubset$start)
        TADsubset$end <- as.numeric(TADsubset$end)
        priorTADsubset <- TADsubset %>%
            filter(start <= rowStart & rowStart <= end &
                (start != rowStart | end != rowEnd)) %>% arrange(start)
        nextTADsubset <- TADsubset %>%
            filter(start <= rowEnd & rowEnd <= end &
                (start != rowStart | end != rowEnd)) %>% arrange(-end)
        completeOverlapSubset <- TADsubset %>%
            filter(start <= rowStart & rowEnd <= end &
                (start != rowStart | end != rowEnd))
        priorOverlap <- 0
        prior_TAD_index <- rowTADIndex
        nextOverlap <- 0
        next_TAD_index <- rowTADIndex
        completeOverlap <- FALSE
        if (nrow(priorTADsubset)) {
            priorOverlap <- priorTADsubset %>%
                mutate(inner_TAD_Length = end - rowStart,
                    outer_TAD_Length = end - start,
                    overlap = (inner_TAD_Length / outer_TAD_Length) * 100) %>%
                arrange(-overlap)
            prior_TAD_index <- priorOverlap$TAD_index[1]
            priorOverlap <- priorOverlap$overlap[1]
            if(is.na(prior_TAD_index)) stop(paste("The following TAD has an issue:", rowTADIndex))
        }
        if (nrow(nextTADsubset)) {
            nextOverlap <- nextTADsubset %>%
                mutate(inner_TAD_Length = rowEnd - start,
                    outer_TAD_Length = end - start,
                    overlap = (inner_TAD_Length / outer_TAD_Length) * 100) %>%
                arrange(-overlap)
            next_TAD_index <- nextOverlap$TAD_index[1]
            nextOverlap <- nextOverlap$overlap[1]
            if(is.na(next_TAD_index)) stop(paste("The following TAD has an issue:", rowTADIndex))
        }
        if (nrow(completeOverlapSubset)) { completeOverlap <- TRUE }
        return(list(prior=as.character(priorOverlap), subsequent=as.character(nextOverlap), com=as.character(completeOverlap), prior_tad=as.character(prior_TAD_index), next_tad=as.character(next_TAD_index)))
    }
    
    merge_TADs <- function(x, inputDF, cutoff = ${overlap_cutoff}){
        rowChr <- x["chr"]
        rowStart <- as.numeric(x["start"])
        rowEnd <- as.numeric(x["end"])
        rowTADIndex <- as.character(x['TAD_index'])
        rowPriorOverlap <- as.double(x["prior_overlap"])
        rowPriorTADIndex <- as.character(x["prior_TAD_index"])
        rowNextOverlap <- as.double(x["next_overlap"])
        rowNextTADIndex <- as.character(x["next_TAD_index"])
        newStart <- rowStart
        newEnd <- rowEnd
        if (rowNextOverlap >= cutoff && rowNextTADIndex != rowTADIndex) {
            newEnd <- inputDF %>% filter(TAD_index == rowNextTADIndex)
            if(is.na(newEnd %>% pull(end) %>% length())) print(newEnd$end[1])
            newEnd <- newEnd$end[1]
        }
        if (rowPriorOverlap >= cutoff & rowPriorTADIndex != rowTADIndex) {
            newStart <- inputDF %>% filter(TAD_index == rowPriorTADIndex)
            newStart <- newStart$start[1]
        }
        return(list(newstart=newStart, newend=newEnd))
    }
    
    recursive_merge <- function(tadDF){
        tadDF$TAD_index <- paste0('TAD_', seq(1, nrow(tadDF)))
        overlap_results <- apply(tadDF, 1, find_TAD_overlap, tadDF)
        tadDF$prior_overlap <- as.double(lapply(overlap_results, "[[", 'prior'))
        tadDF$prior_TAD_index <- as.character(lapply(overlap_results, "[[", 'prior_tad'))
        tadDF$next_overlap <- as.double(lapply(overlap_results, "[[", 'subsequent'))
        tadDF$next_TAD_index <- as.character(lapply(overlap_results, "[[", 'next_tad'))
        tadDF$complete_overlap <- as.logical(lapply(overlap_results, "[[", 'com'))
        merge_results <- apply(tadDF, 1, merge_TADs, tadDF, ${overlap_cutoff})
        tadDF$end <- as.numeric(lapply(merge_results, "[[", 'newend'))
        tadDF$start <- as.numeric(lapply(merge_results, "[[", 'newstart'))
        candidateDF <- tadDF %>% distinct(chr, start, end, .keep_all=TRUE)
        if (nrow(tadDF) == nrow(candidateDF)) {
            return(candidateDF)
        } else {
            candidateDF$TAD_index <- paste0('TAD_', seq(1, nrow(candidateDF)))
            overlap_results <- apply(candidateDF, 1, find_TAD_overlap, candidateDF)
            candidateDF$prior_overlap <- as.double(lapply(overlap_results, "[[", 'prior'))
            candidateDF$prior_TAD_index <- as.character(lapply(overlap_results, "[[", 'prior_tad'))
            candidateDF$next_overlap <- as.double(lapply(overlap_results, "[[", 'subsequent'))
            candidateDF$next_TAD_index <- as.character(lapply(overlap_results, "[[", 'next_tad'))
            candidateDF$complete_overlap <- as.logical(lapply(overlap_results, "[[", 'com'))
            candidateDF <- candidateDF %>% filter(complete_overlap == FALSE)
            candidateDF$TAD_index <- paste0('TAD_', seq(1, nrow(candidateDF)))
            overlap_results <- apply(candidateDF, 1, find_TAD_overlap, candidateDF)
            candidateDF$prior_overlap <- as.double(lapply(overlap_results, "[[", 'prior'))
            candidateDF$prior_TAD_index <- as.character(lapply(overlap_results, "[[", 'prior_tad'))
            candidateDF$next_overlap <- as.double(lapply(overlap_results, "[[", 'subsequent'))
            candidateDF$next_TAD_index <- as.character(lapply(overlap_results, "[[", 'next_tad'))
            candidateDF$complete_overlap <- as.logical(lapply(overlap_results, "[[", 'com'))
            return(recursive_merge(candidateDF))
        }
    }
    
    # ---- Step i. Manage TAD redundancy ----
    general_TAD_DF <- read_tsv(${_input["tad"]:r}, col_names=c('chr', 'start', 'end'), show_col_types = FALSE)
    general_TAD_DF <- general_TAD_DF[with(general_TAD_DF, order(chr, start, -end)),]
    final_brain_TAD_DF <- recursive_merge(general_TAD_DF)
    formatted_final_DF <- final_brain_TAD_DF %>% filter(chr %in% paste0('chr', seq(1,22))) %>% subset(select=c("chr", "start", "end"))
    formatted_final_DF$end <- as.numeric(formatted_final_DF$end)
    formatted_final_DF$start <- as.numeric(formatted_final_DF$start)
    formatted_final_DF <- final_brain_TAD_DF %>% subset(select=c("chr", "start", "end"))
    formatted_final_DF$start <- format(formatted_final_DF$start, scientific = FALSE)
    formatted_final_DF$end <- format(formatted_final_DF$end, scientific = FALSE)
    write.table(formatted_final_DF, ${_output["gentad"]:r}, sep="\t", append=FALSE, row.names=FALSE, col.names=FALSE, quote=FALSE)
    
    # ---- Step ii. Generate TADB-enhanced cis windows and extended TADBs ----
    general <- read_tsv(${_output["gentad"]:r}, col_names = c("chr", "start", "end")) %>% mutate(row_index = row_number())
    specific <- read_tsv(${_input["tad"]:r}, col_names = c("chr", "start", "end")) %>% mutate(row_index = row_number())
    
    chromosomes <- paste0('chr', seq(1:22))
    chromosomes <- c(chromosomes,"chrX")
    bplengths <- c(248956422, 242193529, 198295559, 190214555, 181538259, 170805979, 159345973, 145138636,
                   138394717, 133797422, 135086622,
                   133275309, 114364328, 107043718, 101991189, 90338345, 83257441,
                   80373285, 58617616, 64444167, 46709983, 50818468, 156040895)
    chrDF <- data.frame(chr = chromosomes, left = bplengths)
    
    find_left_boundary = function(chrm, start_pos, end_pos){
        within_TAD = specific %>% filter(chr == chrm & start >= start_pos & end <= end_pos)
        within_TAD_num = nrow(within_TAD)
        if(within_TAD_num == 1){
            left = within_TAD %>% pull(start) %>% as.integer()
        }else{
            TAD_start_list = within_TAD %>% pull(start) %>% sort()
            left = TAD_start_list[2] %>% as.integer()
        }
        return(left)
    }
    left_table = general %>% mutate(left = pmap(list(chrm = chr, start_pos = start, end_pos = end), find_left_boundary)) %>% select(chr, left) %>% mutate(left = as.integer(left))
    left_table_final = rbind(left_table, chrDF)
    
    find_right_boundary = function(chrm, start_pos, end_pos){
        within_TAD = specific %>% filter(chr == chrm & start >= start_pos & end <= end_pos)
        within_TAD_num = nrow(within_TAD)
        if(within_TAD_num == 1){
            right = within_TAD %>% pull(end) %>% as.integer()
        }else{
            TAD_end_list = within_TAD %>% pull(end) %>% sort(decreasing = T)
            right = TAD_end_list[2] %>% as.integer()
        }
        return(right)
    }
    right_table = general %>% mutate(right = pmap(list(chrm = chr, start_pos = start, end_pos = end), find_right_boundary)) %>% select(chr, right) %>% mutate(right = as.integer(right))
    chr_start = tibble(chr = chromosomes, right = 0)
    right_table_final = rbind(right_table, chr_start)
    
    find_next_boundary = function(chrm, end_pos){
        left_boundaries = left_table_final %>% filter(chr == chrm, left >= end_pos) %>% pull(left) %>% sort()
        first_left = left_boundaries[1]
        return(first_left)
    }
    find_previous_boundary = function(chrm, start_pos){
        right_boundaries = right_table_final %>% filter(chr == chrm, right <= start_pos) %>% pull(right) %>% sort(decreasing = T)
        first_right = right_boundaries[1]
        return(first_right)
    }
    TADB = general %>% mutate(previous_bound = pmap(list(chr, start), find_previous_boundary),
        next_bound = pmap(list(chr, end), find_next_boundary)) %>%
        select(-start, -end) %>% rename(start = previous_bound) %>%
        rename(end = next_bound) %>% mutate(start = as.integer(start), end = as.integer(end)) %>%
        select(chr, start, end)
    TADB = TADB %>% distinct()
    
    if_fully_cover = function(chrm, start_pos, end_pos){
        potential = TADB %>% filter(chr == chrm, start <= start_pos, end >= end_pos)
        if(nrow(potential) <= 1){
            return(0)
        }else{
            return(1)
        }
    }
    TADB_final = TADB %>% mutate(cover_status = pmap(list(chrm = chr, start_pos = start, end_pos = end), if_fully_cover)) %>%
        filter(cover_status != 1) %>% mutate(index = paste0("TADB", row_number())) %>% select(- cover_status) %>%
        rename(`#chr` = chr)
    write_tsv(TADB_final, ${_output["gentadb"]:r})
    
    # ---- gene cis windows ----
    all_gene <- read_tsv(${_input["gene"]:r})
    all_gene = all_gene %>% select(-`...1`, -gene_name) %>% rename(chr = `#chr`, gene_start = start, gene_end = end)  %>% distinct()
    TADB_final = TADB_final %>% rename(chr = "#chr")
    gene_TADB_tb = left_join(all_gene, TADB_final, by = "chr", relationship = "many-to-many") %>%
        filter(!(gene_start > end) &  !(gene_end < start))
    
    # extended TADB-enhanced cis windows
    cis_TADB = left_join(gene_TADB_tb, chrDF, by = "chr") %>% rename(chr_end = left) %>%
        mutate(cis_start = pmax(0, gene_start - 1000000)) %>%
        mutate(cis_end = pmin(chr_end, gene_end + 1000000))
    extended =  cis_TADB %>%
        mutate(true_start = pmin(start, cis_start),
               true_end = pmax(end, cis_end)) %>%
        arrange(chr, true_start) %>% ungroup() %>%
        select(chr, gene_id, true_start, true_end) %>%
        rename(start = true_start, end = true_end)
    extended_final = extended %>% group_by(chr, gene_id) %>% summarize(start = min(start), end = max(end)) %>% arrange(chr, start)
    ordered_chr <- c(paste0("chr", as.character(1:22)), "chrX")
    extend_cis_ordered <- extended_final %>% mutate(chr = factor(chr, levels = ordered_chr)) %>%
        arrange(chr) %>% rename(`#chr` = chr) %>% select(`#chr`, start, end, gene_id)
    write_tsv(extend_cis_ordered, ${_output["cis"]:r})
    
    # ---- extended TADB ----
    cis_TADB = left_join(gene_TADB_tb, chrDF, by = "chr") %>% rename(chr_end = left) %>%
        mutate(cis_start = pmax(0, gene_start - 1000000)) %>%
        mutate(cis_end = pmin(chr_end, gene_end + 1000000))
    cis_summary = cis_TADB %>% group_by(index) %>% summarize(min_start = min(cis_start), max_end = max(cis_end))
    extend_TADB = left_join(cis_summary, TADB_final, by = "index") %>% mutate(new_start = pmin(start, min_start)) %>%
        mutate(new_end = pmax(end, max_end)) %>% mutate(length = new_end - new_start) %>%
        mutate(old_length = end - start) %>% mutate(length_diff = length - old_length) %>% select(chr, new_start, new_end) %>%
        rename(start = new_start, end = new_end) %>% mutate(index = paste0("TADB", row_number()))
    extend_TADB <- extend_TADB %>% mutate(chr = factor(chr, levels = ordered_chr)) %>%
        arrange(chr, start) %>% rename(`#chr` = chr) %>% mutate(index = paste0("TADB_", row_number()))
    write_tsv(extend_TADB, ${_output["ext"]:r})
    
    cat("DONE: generalized TADs =", nrow(formatted_final_DF), "| TADB_final =", nrow(TADB_final), "| cis windows =", nrow(extend_cis_ordered), "| extended TADB =", nrow(extend_TADB), "\n")

## Troubleshooting

| Problem | Likely cause | Fix |
| --- | --- | --- |
| `Error: ... object 'specific' not found` | TAD input not read | Check `--tad-input` path and that it is tab-separated with 3 columns |
| Empty / tiny cis-window output | gene table has no rows on the input chromosomes | Ensure `--gene-coords` covers the same chromosomes as `--tad-input` |
| R package errors (`tidyverse`) | `tidyverse` not installed | Install `tidyverse` in the active R environment |
| Very long runtime | many TADs / genome-wide input | The recursive merge is O(n^2) per round; subset by chromosome for a quick test |

## Anticipated Results

The pipeline produces output files in the `output/` subdirectory named after the workflow step. Verify success by checking that output files exist and are non-empty. See the **Output** section above for the expected file names and formats.